# 01 Baseline CNN from scratch

Experiment 1. Trains `BaselineCNN` on the driver-wise split (the split for every number in the results table). Run on a Colab GPU after cloning the repo and unzipping the data into `data/` (see README).

In [ ]:
import json, os, matplotlib.pyplot as plt
import torch
from data import make_loaders
from models import build_model, count_parameters
from train import fit, set_seed, get_device
from eval import evaluate, summarize, per_class_report, accuracy_by_driver, plot_confusion_matrix

os.makedirs('results', exist_ok=True)
set_seed()
device = get_device()
loaders = make_loaders('splits/driverwise.csv', data_dir='data')
model = build_model('baseline')
print(device, f'{count_parameters(model):,} parameters')

In [ ]:
history = fit(model, loaders, epochs=15, lr=1e-3, device=device, ckpt_path='checkpoints/baseline_driverwise.pt')

## Learning curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val'); ax[0].set_title('loss')
ax[1].plot(history['train_acc'], label='train'); ax[1].plot(history['val_acc'], label='val'); ax[1].set_title('accuracy')
for a in ax: a.set_xlabel('epoch'); a.legend()
fig.tight_layout(); fig.savefig('figures/baseline_curves.png', dpi=150)

## Test results (driver-wise)

In [ ]:
test = evaluate(model, loaders['test'], device)
print(summarize(test))
print(per_class_report(test))
print(accuracy_by_driver(test, loaders['test'].dataset.samples['subject']))

In [ ]:
plot_confusion_matrix(test, 'figures/baseline_confusion.png');

In [ ]:
json.dump({'history': history, 'test': summarize(test), 'by_driver': accuracy_by_driver(test, loaders['test'].dataset.samples['subject'])}, open('results/baseline_driverwise.json', 'w'), indent=2, default=float)